In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

In [ ]:
class ChatbotState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatbotState):
    messages = state['messages']

    response = llm.invoke(messages)

    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()

In [ ]:
graph = StateGraph(ChatbotState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id = 1

while True:
    user_message = input('Type here: ')

    print(f"Me: {user_message}")

    if user_message.strip().lower() in ['exit']:
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = workflow.invoke({'messages': [HumanMessage(content=user_message)]}, config = config)

    print(f"AI: {response['messages'][-1].content[0]['text']}")